In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

windspeed_df = pd.read_csv(str(ROOT / "data/DMI_ws.csv"))
irr_df = pd.read_csv(str(ROOT / "data/DMI_radiation.csv"))
PV_df = pd.read_csv(str(ROOT / "data/SyslabPV_15min_collective_PV.csv"))
PV_df["ts"] = pd.to_datetime(PV_df['ts'])
PV_df = PV_df.set_index("ts")
PV_df = PV_df
Wind_df = pd.read_csv(str(ROOT / "data/SyslabWind_15min_nozeros.csv"))
df = pd.read_csv(str(ROOT / "data/combined_data_positive_gen.csv"))

In [ ]:
seasonality = 24*4 
PV_df["Power diff"] = PV_df['Collective PV'].diff(periods=seasonality)

In [ ]:
result = seasonal_decompose(PV_df['Collective PV'].dropna(), model='addiditve', period=seasonality)
trend = result.trend.dropna()
seasonal = result.seasonal.dropna()
residual = result.resid.dropna()

In [ ]:

# Plot the decomposed components
plt.figure(figsize=(6,6))
#x_labels = [t.strftime('%m %Y') for t in PV_df.index]
plt.subplot(4, 1, 1)
plt.plot(PV_df.index,PV_df['Collective PV'], label='Original Series')

#plt.xticks(range(0, PV_df.size, 2880), x_labels[::2880], rotation=45)
plt.legend()

plt.subplot(4, 1, 2)
plt.plot(trend, label='Trend')
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(seasonal, label='Seasonal')
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(residual, label='Residuals')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
windspeed_df = pd.read_csv(str(ROOT / "data/DMI_ws.csv"))
irr_df = pd.read_csv(str(ROOT / "data/DMI_radiation.csv"))
irr_df["ts"] = pd.to_datetime(irr_df['ts'])
irr_df = irr_df.set_index("ts")


PV_df = pd.read_csv(str(ROOT / "data/SyslabPV_60min.csv"))
PV_df["ts"] = pd.to_datetime(PV_df['ts'])
PV_df = PV_df.set_index("ts")


Wind_df = pd.read_csv(str(ROOT / "data/SyslabWind_15min_nozeros.csv"))
df = pd.read_csv(str(ROOT / "data/combined_data_positive_gen.csv"))


y = PV_df["PV B715"]        # target (has NaNs)
X = irr_df[["mean radiation"]]  # exogenous (should NOT have NaNs ideally)

In [ ]:
X

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def Sarimax(file_path,endo_name,exo_name,granularity=10):
    """Using the SARIMAX function to fill gaps in the endogenous variable using the exogeneous variable
    Granularity is the amount of time between measurements in minutes"""


    try:
        df = pd.read_csv(file_path,delimiter=",")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    
    endo = df[endo_name]
    exo = df[exo_name]

    seasonality = 24*60/granularity
    model = SARIMAX(
        endo,
        exog=exo,
        order=(1,1,1),
        seasonal_order=(1,1,1,seasonality),  # daily seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit()

    endo_modelled = results.fittedvalues
    endo_filled = endo.copy()
    endo_filled[endo.isna()] = endo_modelled[endo.isna()]

    results.plot_diagnostics()

    endo_filled.to_csv(f'{file_path[:-4]}_SARIMAX.csv', index=True)